In [24]:
# Janela Qt: widgets precisam de backend interativo.
%matplotlib qt
import matplotlib.pyplot as plt
from matplotlib.widgets import CheckButtons, RadioButtons, Button
import numpy as np
import pandas as pd

In [27]:
# Iris: coluna "target" = espécie; eixo X fixo em comprimento da pétala; o rádio só troca o Y.
# Fecha figuras antigas para evitar clicar no rádio de uma janela Qt "velha" (nada parece mudar na janela nova).
plt.close("all")

df = pd.read_csv("iris_dataset.csv")
cores = {"setosa": "#1f77b4", "versicolor": "#2ca02c", "virginica": "#d62728"}

fig, ax = plt.subplots(figsize=(10, 6))
# Margens em fração da figura: área livre à esquerda para Check/Radio/Button sem cobrir o scatter.
plt.subplots_adjust(left=0.28, bottom=0.18)

# Um scatter por espécie — dicionário nome → PathCollection — para ligar/desligar cada curva no checkbox.
scatters = {}
for sp, grp in df.groupby("target"):
    scatters[sp] = ax.scatter(
        grp["petal length (cm)"],
        grp["petal width (cm)"],
        label=sp,
        color=cores[sp],
        alpha=0.8,
        edgecolor="black",
        linewidths=0.4,
    )

ax.set_title(
    "Iris interativo — ligue/desligue espécies, troque eixos", fontsize=14, pad=30
)
ax.text(
    0.5,
    1.02,
    "CheckButtons + RadioButtons + Button (reset)",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
    color="#555555",
)
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.legend(loc="lower right")

# --- CheckButtons: caixas independentes (várias podem ficar marcadas) ---
# Rótulos na mesma ordem das chaves de scatters; estado inicial = todas visíveis.
ax_check = plt.axes([0.02, 0.70, 0.18, 0.25])
check = CheckButtons(ax_check, list(scatters.keys()), [True, True, True], useblit=False)


def toggle(label):
    # label é o nome da espécie cuja caixa foi clicada; só alternamos visibilidade desse scatter.
    scatters[label].set_visible(not scatters[label].get_visible())
    fig.canvas.draw()


check.on_clicked(toggle)

# useblit=False: com True, o widget pode redesenhar só o painel do rádio e o scatter principal não atualiza.
ax_radio = plt.axes([0.02, 0.35, 0.18, 0.3])
features_y = ["petal width (cm)", "sepal length (cm)", "sepal width (cm)"]
radio = RadioButtons(ax_radio, features_y, useblit=False)


def trocar_y(_label=None):
    # Usa o estado do widget (sempre coerente com o botão marcado), não só o argumento do callback.
    feat = radio.value_selected
    for sp, grp in df.groupby("target"):
        xcol = grp["petal length (cm)"].to_numpy()
        ycol = grp[feat].to_numpy()
        scatters[sp].set_offsets(np.column_stack([xcol, ycol]))
    ax.set_ylabel(feat)
    pts = np.vstack([sc.get_offsets() for sc in scatters.values()])
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    pad = 0.05 * max(x1 - x0, y1 - y0, 1e-9)
    ax.set_xlim(x0 - pad, x1 + pad)
    ax.set_ylim(y0 - pad, y1 + pad)
    # Redesenho completo (draw_idle às vezes não atualiza o eixo principal após set_offsets no Qt).
    fig.canvas.draw()


radio.on_clicked(trocar_y)

# --- Button: dispara uma ação ao clicar (não guarda “ligado/desligado”) ---
ax_btn = plt.axes([0.02, 0.20, 0.18, 0.08])
btn = Button(ax_btn, "Reset", color="#e6e6e6", hovercolor="#cccccc")


def resetar(event):
    for sc in scatters.values():
        sc.set_visible(True)
    radio.set_active(0)
    fig.canvas.draw()


btn.on_clicked(resetar)
plt.savefig("buttons.png", dpi=1000)
plt.show()